# First prototype for HyperBubble Resolution-Oriented GNN

In [10]:
from pathlib import Path
from typing import Any, Dict, List, Optional
import json
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

In [11]:
NUC2ID = { 'A':1, 'C':2, 'G':3, 'T':4, 'N':5 }
def seq_to_tokens(seq: str) -> torch.LongTensor:
    return torch.tensor([NUC2ID.get(c, 5) for c in seq], dtype=torch.long)

def _safe_get(d: Dict[str, Any], key: str, default):
    v = d.get(key, default)
    return default if v is None else v

ID2NUC = {v:k for k,v in NUC2ID.items()}

def tokens_to_seq(tokens: torch.LongTensor) -> str:
    return "".join(ID2NUC.get(int(t), "N") for t in tokens.tolist())

In [12]:
class HyperbubbleDataset(Dataset):
    """
    Minimal GNN-ready dataset from your JSONL:
      - Node features:
          seq_tokens : [N, K] Long (embed this; no one-hot)
          x_cov      : [N, 1] Float (KC coverage; 0 if unknown)
      - Graph:
          edge_index : [2, E] Long
          edge_attr  : [E, 5] Float (len_nodes, len_bp, cov_min, cov_mean, on_ref)
          start_idx, end_idx : Long
      - Labels:
          y_edge_mask : [E] Long (1 on labeled path edges, else 0)
          label_path_idx : Long (-1 if none)
    """
    def __init__(self, files: List[str]):
        self.files = [Path(p) for p in files]
        self.records: List[Dict[str, Any]] = []
        for fp in self.files:
            with fp.open("r") as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        self.records.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue

    def __len__(self) -> int:
        return len(self.records)

    def _build_graph(self, rec: Dict[str, Any]) -> Data:
        # --- nodes (map seq -> idx), carry best-known coverage ---
        node_seqs: Dict[str, int] = {}
        node_covs: List[float] = []

        def ensure_node(seq: str, cov: Optional[float] = None) -> int:
            if seq not in node_seqs:
                node_seqs[seq] = len(node_seqs)
                node_covs.append(float(cov) if cov is not None else 0.0)
            else:
                i = node_seqs[seq]
                if cov is not None and node_covs[i] == 0.0:
                    node_covs[i] = float(cov)
            return node_seqs[seq]

        # endpoints
        start_seq = rec["start_seq"]
        end_seq   = rec["end_seq"]

        # nodes with cov
        for n in _safe_get(rec, "nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))

        # include 1-hop context nodes (optional cov)
        for n in _safe_get(rec, "upstream_nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))
        for n in _safe_get(rec, "downstream_nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))

        # make sure endpoints are present
        ensure_node(start_seq)
        ensure_node(end_seq)

        # --- edges + attributes ---
        edge_src, edge_dst, edge_attr = [], [], []
        for e in _safe_get(rec, "edges", []):
            u = ensure_node(e["source_seq"])
            v = ensure_node(e["target_seq"])
            edge_src.append(u)
            edge_dst.append(v)
            edge_attr.append([
                float(e.get("len_nodes", 0)),
                float(e.get("len_bp", 0)),
                float(e.get("cov_min", 0)),
                float(e.get("cov_mean", 0.0)),
                1.0 if e.get("on_ref", False) else 0.0
            ])

        # --- node features ---
        node_order = [None] * len(node_seqs)
        for s, idx in node_seqs.items():
            node_order[idx] = s

        # k is constant across the dataset -> sequences are k-mers
        seq_tokens = torch.stack([seq_to_tokens(s) for s in node_order], dim=0)  # [N, K]
        x_cov = torch.tensor(node_covs, dtype=torch.float32).unsqueeze(1)        # [N, 1]

        start_idx = torch.tensor(node_seqs[start_seq], dtype=torch.long)
        end_idx   = torch.tensor(node_seqs[end_seq], dtype=torch.long)

        edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
        edge_attr  = (torch.tensor(edge_attr, dtype=torch.float32)
                      if edge_attr else torch.zeros((0,5), dtype=torch.float32))

        # --- labels from label_path ---
        num_edges = edge_index.size(1)
        y_edge_mask = torch.zeros(num_edges, dtype=torch.long)
        label_path_idx = -1
        paths_list = _safe_get(rec, "paths", [])
        lp = rec.get("label_path")  # may be None
        if lp is not None and 0 <= lp < len(paths_list) and num_edges > 0:
            label_path_idx = int(lp)
            y_edge_mask[torch.tensor(paths_list[label_path_idx], dtype=torch.long)] = 1

        data = Data(
            # node features
            seq_tokens=seq_tokens,    # Long [N, K]
            x_cov=x_cov,              # Float [N, 1]
            # graph
            edge_index=edge_index,    # Long [2, E]
            edge_attr=edge_attr,      # Float [E, 5]
            start_idx=start_idx,      # Long []
            end_idx=end_idx,          # Long []
            num_nodes=seq_tokens.size(0),
            # labels
            y_edge_mask=y_edge_mask,                      # Long [E]
            label_path_idx=torch.tensor(label_path_idx)   # Long []
        )
        # keep only tensors in Data to avoid collation issues
        # (bubble_id/k useful but optional)
        if "bubble_id" in rec:
            data.bubble_id = torch.tensor(rec["bubble_id"], dtype=torch.long)
        if "k" in rec:
            data.k = torch.tensor(rec["k"], dtype=torch.long)
        return data

    def __getitem__(self, idx: int) -> Data:
        return self._build_graph(self.records[idx])


In [13]:
from torch.utils.data import Subset

# Build the full dataset first
dataset = HyperbubbleDataset(jsonl_paths)

# Keep only items where label_path_idx >= 0
labeled_indices = []
for i in range(len(dataset)):
    d = dataset[i]
    lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
    if lp >= 0:
        labeled_indices.append(i)

dataset = Subset(dataset, labeled_indices)

# ---- train/test split ----
import torch
from torch.utils.data import random_split
from torch_geometric.loader import DataLoader

test_ratio = 0.2
n_total = len(dataset)
n_test = max(1, int(round(n_total * test_ratio))) if n_total > 0 else 0
n_train = n_total - n_test

train_dataset, test_dataset = random_split(
    dataset,
    [n_train, n_test],
    generator=torch.Generator().manual_seed(42)
)

# Peek (from train split)
sample = train_dataset[0]
print("num_nodes:", sample.num_nodes)
print("num_edges:", sample.edge_index.size(1))

for i, (tok_row, cov) in enumerate(zip(sample.seq_tokens, sample.x_cov)):
    seq = tokens_to_seq(tok_row)
    print(f"node {i}: seq={seq}, cov={cov.item()}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=0)


num_nodes: 12
num_edges: 4
node 0: seq=CAGGCCGACTCGACCAGTGAG, cov=425.0
node 1: seq=TAGCCCCGTTACATCTTCCGC, cov=396.0
node 2: seq=CGACTCGACCAGTGAGCTATT, cov=414.0
node 3: seq=TGGTTTAGCCCCGTTACATCT, cov=384.0
node 4: seq=TTAGCCCCGTTACATCTTCCG, cov=392.0
node 5: seq=TTTAGCCCCGTTACATCTTCC, cov=389.0
node 6: seq=GTTTAGCCCCGTTACATCTTC, cov=390.0
node 7: seq=GGTTTAGCCCCGTTACATCTT, cov=388.0
node 8: seq=AGGCCGACTCGACCAGTGAGC, cov=424.0
node 9: seq=GGCCGACTCGACCAGTGAGCT, cov=423.0
node 10: seq=GCCGACTCGACCAGTGAGCTA, cov=422.0
node 11: seq=CCGACTCGACCAGTGAGCTAT, cov=419.0


In [14]:
# --- device selection (CPU / CUDA / DirectML) ---
USE_DIRECTML = True  # set False to stay on CPU; CUDA will be auto-used if available and USE_DIRECTML is False

import torch
device = torch.device('cpu')
if not USE_DIRECTML and torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_DIRECTML:
    try:
        import torch_directml
        device = torch_directml.device()
        print("Using DirectML:", device)
    except Exception as e:
        print("DirectML not available, falling back to CPU:", e)
        device = torch.device('cpu')

# --- simple embedding + dense GCN that avoids PyG custom ops ---
import torch.nn as nn
import torch.nn.functional as F

class DenseGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)

    def forward(self, H, edge_index_local, n_nodes: int):
        if n_nodes == 0:
            return H
        A = H.new_zeros((n_nodes, n_nodes))
        if edge_index_local.numel() > 0:
            src, dst = edge_index_local
            one = torch.ones_like(src, dtype=H.dtype)
            A.index_put_((src, dst), one, accumulate=True)

        # add self-loops w/o torch.eye
        idx = torch.arange(n_nodes, device=H.device)
        try:
            A[idx, idx] += 1
        except Exception:
            flat = A.view(-1)
            diag_idx = torch.arange(0, n_nodes*n_nodes, step=n_nodes+1, device=H.device)
            flat.index_add_(0, diag_idx, torch.ones(n_nodes, device=H.device, dtype=H.dtype))
            A = flat.view(n_nodes, n_nodes)

        deg = A.sum(dim=1) + 1e-8
        D_inv_sqrt = deg.pow(-0.5)
        A_norm = (D_inv_sqrt[:, None] * A) * D_inv_sqrt[None, :]
        return A_norm @ self.lin(H)

# ---- Simple GNN with sequence embedding + GCN + edge MLP ----
class HyperbubbleGNN(nn.Module):
    def __init__(
        self,
        vocab_size=5,          # tokens: 0=PAD, A,C,G,T (and optionally N)
        embed_dim=32,          # per-token embedding dim
        gcn_hidden=64,         # node hidden dim (consistent throughout)
        edge_hidden=64,        # hidden in edge MLP
        edge_feat_dim=5,       # [len_nodes, len_bp, cov_min, cov_mean, on_ref]
        use_lstm=False,        # optional: token encoder = mean or BiLSTM
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.use_lstm = use_lstm
        if use_lstm:
            self.lstm = nn.LSTM(embed_dim, gcn_hidden // 2, batch_first=True, bidirectional=True)
            self.node_in = gcn_hidden + 1   # + coverage scalar
        else:
            self.node_in = embed_dim + 1

        self.gcn_hidden = gcn_hidden
        self.gcn1 = DenseGCNLayer(self.node_in, gcn_hidden)
        self.gcn2 = DenseGCNLayer(gcn_hidden, gcn_hidden)

        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * gcn_hidden + edge_feat_dim, edge_hidden),
            nn.ReLU(),
            nn.Linear(edge_hidden, 1),
        )

    def encode_nodes(self, seq_tokens, x_cov):
        """
        seq_tokens: [N, K] tokenized k-mer
        x_cov     : [N, 1]
        returns   : [N, node_in]
        """
        E = self.embed(seq_tokens)
        mask = (seq_tokens != 0).float().unsqueeze(-1)
        if self.use_lstm:
            lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1).long()
            mean_init = (E * mask).sum(dim=1) / lengths.clamp(min=1).unsqueeze(-1).float()
            H0, _ = self.lstm(E)
            Hseq = (H0 * mask).sum(dim=1) / lengths.clamp(min=1).unsqueeze(-1).float()
        else:
            lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1)
            Hseq = (E * mask).sum(dim=1) / lengths.unsqueeze(-1)

        X = torch.cat([Hseq, x_cov], dim=1)
        return X

    def forward(self, data):
        """
        data: PyG batch with fields:
          - seq_tokens [N,K], x_cov [N,1], edge_index [2,E], edge_attr [E,5],
            batch [N] (graph ids), num_nodes, etc.
        returns:
          logits [E]  (edge-wise)
        """
        device = data.seq_tokens.device
        N = data.num_nodes
        E = data.edge_index.size(1)

        X0 = self.encode_nodes(data.seq_tokens, data.x_cov)

        out_node = X0.new_zeros((N, self.gcn_hidden))
        batch_vec = data.batch if hasattr(data, "batch") else torch.zeros(N, dtype=torch.long, device=device)
        num_graphs = int(batch_vec.max().item()) + 1 if N > 0 else 0

        for g in range(num_graphs):
            node_ids = (batch_vec == g).nonzero(as_tuple=False).view(-1)
            n_nodes = int(node_ids.numel())
            if n_nodes == 0:
                continue

            local_map = torch.full((N,), -1, dtype=torch.long, device=device)
            local_map[node_ids] = torch.arange(n_nodes, device=device, dtype=torch.long)

            ei = data.edge_index
            keep = (local_map[ei[0]] >= 0) & (local_map[ei[1]] >= 0)
            keep_idx = keep.nonzero(as_tuple=False).view(-1)
            if keep_idx.numel() == 0:
                H = F.relu(self.gcn1(X0[node_ids], edge_index_local=torch.empty(2,0,device=device,dtype=torch.long), n_nodes=n_nodes))
                H = F.relu(self.gcn2(H,          edge_index_local=torch.empty(2,0,device=device,dtype=torch.long), n_nodes=n_nodes))
                out_node[node_ids] = H
                continue

            src_local = local_map[ei[0, keep_idx]]
            dst_local = local_map[ei[1, keep_idx]]
            edge_local = torch.stack([src_local, dst_local], dim=0)

            H = F.relu(self.gcn1(X0[node_ids], edge_local, n_nodes))
            H = F.relu(self.gcn2(H,            edge_local, n_nodes))
            out_node[node_ids] = H

        if E == 0:
            return torch.empty((0,), device=device)

        u, v = data.edge_index
        U = out_node[u]
        V = out_node[v]
        EA = data.edge_attr if hasattr(data, "edge_attr") and data.edge_attr.numel() > 0 \
             else out_node.new_zeros((E, 5))
        feats = torch.cat([U, V, EA], dim=1)
        logits = self.edge_mlp(feats).squeeze(-1)
        return logits

Using DirectML: privateuseone:0


In [15]:
import torch.nn.functional as F

model = HyperbubbleGNN(
    vocab_size=5, embed_dim=32, gcn_hidden=64, edge_hidden=64, edge_feat_dim=5, use_lstm=False
).to(device)

optim = torch.optim.AdamW(model.parameters(), lr=1e-3)

def _num_graphs_in_batch(node_batch):
    return int(node_batch.max().item()) + 1 if node_batch.numel() else 0

def _start_global_idx_for_graph(data, node_batch, g):
    node_ids = (node_batch == g).nonzero(as_tuple=False).view(-1)              # global node ids of graph g
    if node_ids.numel() == 0:
        return None
    start_local = int(data.start_idx[g].item())                                 # local idx within this graph
    return node_ids[start_local]                                                # global idx in the batch

def _candidate_edge_indices_from_start(data, u, node_batch, g, start_global):
    # edges that belong to graph g and start at the graph's start node
    edge_batch = node_batch[u] if u.numel() else u.new_zeros((0,), dtype=torch.long)
    mask = (edge_batch == g) & (u == start_global)
    return mask.nonzero(as_tuple=False).view(-1)

def _gold_choice_within_candidates(data, cand_idx):
    # among candidate edges, the one that lies on the labeled path (should be exactly 1)
    y_mask = data.y_edge_mask[cand_idx]
    pos = (y_mask == 1).nonzero(as_tuple=False).view(-1)
    return pos if pos.numel() == 1 else None

def train_one_epoch_choice(model, loader):
    model.train()
    total_loss, total_bubbles = 0.0, 0

    for data in loader:
        data = data.to(device)
        logits = model(data)                            # [E] scores for every edge in the (batched) graphs
        if logits.numel() == 0:
            continue

        u = data.edge_index[0]                          # [E] source node (global idx in batch)
        node_batch = data.batch                         # [N] graph id per node

        num_graphs = _num_graphs_in_batch(node_batch)
        batch_loss = 0.0
        bubbles_used = 0

        for g in range(num_graphs):
            start_global = _start_global_idx_for_graph(data, node_batch, g)
            if start_global is None:
                continue

            cand_idx = _candidate_edge_indices_from_start(data, u, node_batch, g, start_global)
            if cand_idx.numel() < 2:
                # need at least 2 candidates to form a non-degenerate choice
                continue

            gold_pos = _gold_choice_within_candidates(data, cand_idx)
            if gold_pos is None:
                continue

            # Cross-entropy over the candidate logits (no log-sigmoid here)
            # shape: [1, C] vs target: [1]
            ce = F.cross_entropy(logits[cand_idx].unsqueeze(0), gold_pos.view(1))
            batch_loss = batch_loss + ce
            bubbles_used += 1

        if bubbles_used == 0:
            continue

        batch_loss = batch_loss / bubbles_used
        optim.zero_grad()
        batch_loss.backward()
        optim.step()

        total_loss += batch_loss.item()
        total_bubbles += bubbles_used

    return total_loss / max(total_bubbles, 1)

@torch.no_grad()
def eval_choice(model, loader):
    model.eval()
    total, correct = 0, 0

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        u = data.edge_index[0]
        node_batch = data.batch
        num_graphs = _num_graphs_in_batch(node_batch)

        for g in range(num_graphs):
            start_global = _start_global_idx_for_graph(data, node_batch, g)
            if start_global is None:
                continue

            cand_idx = _candidate_edge_indices_from_start(data, u, node_batch, g, start_global)
            if cand_idx.numel() < 2:
                continue

            gold_pos = _gold_choice_within_candidates(data, cand_idx)
            if gold_pos is None:
                continue

            pred = logits[cand_idx].argmax().item()
            total += 1
            correct += int(pred == gold_pos.item())

    acc = correct / max(total, 1)
    print(f"[choice-eval] bubbles_used={total}  top1_acc={acc:.3f}")
    return acc

In [17]:
EPOCHS = 100
for epoch in range(1, EPOCHS + 1):
    tr_loss = train_one_epoch_choice(model, train_loader)
    print(f"epoch {epoch}: train_choice_loss={tr_loss:.4f}")
    eval_choice(model, test_loader)

epoch 1: train_choice_loss=0.0441
[choice-eval] bubbles_used=62  top1_acc=0.726
epoch 2: train_choice_loss=0.0457
[choice-eval] bubbles_used=62  top1_acc=0.484
epoch 3: train_choice_loss=0.0458
[choice-eval] bubbles_used=62  top1_acc=0.661
epoch 4: train_choice_loss=0.0451
[choice-eval] bubbles_used=62  top1_acc=0.645
epoch 5: train_choice_loss=0.0459
[choice-eval] bubbles_used=62  top1_acc=0.581
epoch 6: train_choice_loss=0.0462
[choice-eval] bubbles_used=62  top1_acc=0.581
epoch 7: train_choice_loss=0.0469
[choice-eval] bubbles_used=62  top1_acc=0.694
epoch 8: train_choice_loss=0.0472
[choice-eval] bubbles_used=62  top1_acc=0.694
epoch 9: train_choice_loss=0.0445
[choice-eval] bubbles_used=62  top1_acc=0.677
epoch 10: train_choice_loss=0.0445
[choice-eval] bubbles_used=62  top1_acc=0.677
epoch 11: train_choice_loss=0.0454
[choice-eval] bubbles_used=62  top1_acc=0.613
epoch 12: train_choice_loss=0.0430
[choice-eval] bubbles_used=62  top1_acc=0.661
epoch 13: train_choice_loss=0.0438
[c